# Histopathologic Cancer Detection — runner
Thin orchestration for **Colab** or **Kaggle**. Heavy logic lives in `src/`.


## 0. Environment detect

In [ ]:
import os, sys, glob, shutil, gzip, zipfile, subprocess
ON_KAGGLE = os.path.exists('/kaggle/input')
print('Kaggle' if ON_KAGGLE else 'Colab/local')

# Diagnostic: show what's actually mounted (self-documents any future failure).
if ON_KAGGLE:
    for d in sorted(glob.glob('/kaggle/input/*')):
        print(' input:', d, '->', sorted(os.path.basename(x) for x in glob.glob(d + '/*'))[:8])

# `kaggle kernels push` uploads only THIS notebook. Bring in src/ + configs from the
# attached dataset (no internet needed). Prefer unzipping code.zip (clean WSI .gz);
# fall back to an already-extracted tree, then git clone (only works with internet).
REPO_URL = 'https://github.com/aRealGem/histopath-cancer-detection'
WORK = '/kaggle/working/repo'
if not os.path.exists('src/train.py'):
    os.makedirs(WORK, exist_ok=True)
    czip = glob.glob('/kaggle/input/**/code.zip', recursive=True)
    extracted = glob.glob('/kaggle/input/**/src/train.py', recursive=True)
    if czip:
        with zipfile.ZipFile(czip[0]) as z:
            z.extractall(WORK)
        print('staged from code.zip:', czip[0])
    elif extracted:
        root = os.path.dirname(os.path.dirname(extracted[0]))
        shutil.copytree(root, WORK, dirs_exist_ok=True)
        print('staged from extracted dataset:', root)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
        print('staged from git clone')
    os.chdir(WORK)

# WSI map: if Kaggle decompressed the shipped .gz into a folder, re-materialize it
# at the config path. (When we unzip code.zip ourselves, the .gz is already intact.)
if not os.path.exists('data/wsi/patch_id_wsi_full.csv.gz'):
    hits = [h for h in glob.glob('/kaggle/input/**/*wsi*full*.csv', recursive=True) if os.path.isfile(h)]
    if hits:
        os.makedirs('data/wsi', exist_ok=True)
        with open(hits[0], 'rb') as fi, gzip.open('data/wsi/patch_id_wsi_full.csv.gz', 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        print('WSI map normalized from', hits[0])

# Offline ImageNet weights: internet is off, so cache the bundled backbone weights.
kmodels = os.path.expanduser('~/.keras/models')
os.makedirs(kmodels, exist_ok=True)
wts = glob.glob('/kaggle/input/**/weights_mobilenet_v3_small_224_1.0_float_no_top_v2.h5', recursive=True)
if wts:
    shutil.copy(wts[0], os.path.join(kmodels, os.path.basename(wts[0])))
    print('cached backbone weights offline')
else:
    print('WARNING: bundled MobileNetV3 weights not found; backbone would need internet.')

# Auto-detect the competition mount and write the real data.root into the configs.
import yaml
cands = glob.glob('/kaggle/input/**/train_labels.csv', recursive=True)
if cands:
    DATA_ROOT = os.path.dirname(cands[0])
    print('data.root ->', DATA_ROOT)
    for cf in ('configs/smoke.yaml', 'configs/baseline.yaml'):
        conf = yaml.safe_load(open(cf)); conf['data']['root'] = DATA_ROOT
        yaml.safe_dump(conf, open(cf, 'w'), sort_keys=False)
else:
    print('WARNING: train_labels.csv not found under /kaggle/input.')

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd(), '| src:', os.path.exists('src/train.py'),
      '| wsi:', os.path.exists('data/wsi/patch_id_wsi_full.csv.gz'))

## 1. Deps (safe on both — TF stays untouched)

In [ ]:
!pip -q install opencv-python-headless pyyaml scikit-learn

## 2. Data
- **Kaggle:** add the competition dataset via the sidebar; it mounts at `/kaggle/input/histopathologic-cancer-detection`. Nothing to download.
- **Colab:** upload `kaggle.json`, then run the download cell.


In [ ]:
if not ON_KAGGLE:
    from pathlib import Path
    Path.home().joinpath('.kaggle').mkdir(exist_ok=True)
    # from google.colab import files; files.upload()  # -> kaggle.json
    # !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !bash scripts/download_data.sh ./data
    # then edit configs/baseline.yaml -> data.root: ./data


## 3. Train -> Evaluate -> Predict

In [ ]:
# smoke.yaml validates the whole pipeline in minutes and writes a real submission.csv.
# For the full run, set HP_CONFIG=configs/baseline.yaml (or edit below) and re-run.
CFG = os.environ.get('HP_CONFIG', 'configs/smoke.yaml')
print('using config:', CFG)
!python -m src.train    --config {CFG}
!python -m src.evaluate --config {CFG}
!python -m src.predict  --config {CFG}

## 4. Inspect ROC + submission

In [ ]:
from IPython.display import Image, display
import pandas as pd, os
if os.path.exists('artifacts/roc_val.png'): display(Image('artifacts/roc_val.png'))
print(pd.read_csv('artifacts/submission.csv').head())

## 5. Submit
Kaggle: use the **Submit** button, or:
```
!kaggle competitions submit -c histopathologic-cancer-detection \
  -f artifacts/submission.csv -m 'MobileNetV3 baseline'
```
Then screenshot the leaderboard for the R09.1 done-criteria.